---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

# ⚖️ Bias-Variance Tradeoff
**ISLP Chapter 2 · Pattern Recognition for the Rest of Us**

> Jump in — each section builds on the last. Run cells top to bottom with Shift+Enter.

### What you'll learn
- What bias and variance mean and why they conflict
- The dart board intuition
- Training vs test MSE as model complexity grows
- Learning curves — diagnosing underfitting vs overfitting

### Time: ~40 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom automatically)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

# ── Load dataset(s) ───────────────────────────────────────────────────────────
try:
    ads = pd.read_csv('https://www.statlearning.com/s/Advertising.csv')
    print(f'✓ Advertising.csv (ISLP): {ads.shape}')
except Exception:
    ads = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Advertising.csv')
    print(f'✓ Advertising.csv (fallback): {ads.shape}')
print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


## 📐 Part 1 — The Dart Board Intuition

In [ ]:
# ── Dart board illustration ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
scenarios = [
    ('High Bias\nLow Variance', [(0.3,0.3),(0.35,0.28),(0.32,0.33),(0.28,0.31),(0.33,0.35),(0.29,0.29)], '#e85d20'),
    ('Low Bias\nHigh Variance', [(-0.1,0.4),(0.5,-0.3),(-0.4,-0.2),(0.3,0.5),(-0.2,-0.4),(0.4,0.1)], '#c0392b'),
    ('High Bias\nHigh Variance', [(0.4,0.3),(0.1,-0.4),(0.5,0.1),(-0.3,0.4),(0.3,-0.2),(0.4,0.5)], '#888'),
    ('Low Bias\nLow Variance ✓', [(0.05,0.03),(-0.04,0.06),(0.02,-0.05),(0.06,0.04),(-0.03,-0.02),(0.04,0.05)], '#1a7a45'),
]

for ax, (title, points, color) in zip(axes, scenarios):
    for r, alpha in [(1,'#f0f0f0'),(0.6,'#e0e8f0'),(0.3,'#c8d8f0'),(0.1,'#a0c0e8')]:
        circle = plt.Circle((0,0), r, color=alpha, zorder=1)
        ax.add_patch(circle)
    ax.plot(0, 0, 'k+', markersize=12, zorder=3, lw=2)
    xs, ys = zip(*points)
    ax.scatter(xs, ys, color=color, s=60, zorder=4, edgecolors='white', lw=0.5)
    ax.set_xlim(-1.2,1.2); ax.set_ylim(-1.2,1.2)
    ax.set_aspect('equal'); ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Bias-Variance as a Dart Board', fontsize=12, y=1.05)
plt.tight_layout()
plt.show()
print("📌 We want bottom-right: low bias (near bullseye) + low variance (tight cluster).")


## 🔬 Part 2 — Training vs Test MSE

In [ ]:
# Show training vs test MSE as model complexity increases
np.random.seed(42)
n_train, n_test = 100, 50
X_all = np.sort(np.random.uniform(0, 4, n_train + n_test))
Y_all = 3 + 2*X_all - 0.5*X_all**2 + np.random.normal(0, 0.8, n_train + n_test)

X_tr, X_te = X_all[:n_train].reshape(-1,1), X_all[n_train:].reshape(-1,1)
Y_tr, Y_te = Y_all[:n_train], Y_all[n_train:]

# Degrees 1-8 — enough to show the full bias-variance tradeoff cleanly
degrees = list(range(1, 9))
train_mse, test_mse = [], []

for d in degrees:
    pipe = Pipeline([('poly', PolynomialFeatures(d)), ('lr', LinearRegression())])
    pipe.fit(X_tr, Y_tr)
    train_mse.append(mean_squared_error(Y_tr, pipe.predict(X_tr)))
    test_mse.append(mean_squared_error(Y_te, pipe.predict(X_te)))

best_degree = degrees[test_mse.index(min(test_mse))]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left — clip y-axis so the U-shape is visible
y_clip = min(test_mse) * 15   # show up to 15x the minimum — keeps the shape clear
axes[0].plot(degrees, train_mse, 'o-', color='#1e5fa8', lw=2.5, label='Training MSE', markersize=7)
axes[0].plot(degrees, test_mse,  'o-', color='#e85d20', lw=2.5, label='Test MSE',     markersize=7)
axes[0].axvline(best_degree, color='#1a7a45', lw=2, ls='--',
                label=f'Sweet spot (degree {best_degree})')
axes[0].set_ylim(0, y_clip)
axes[0].set_xticks(degrees)
axes[0].set_xlabel('Model Complexity (polynomial degree)')
axes[0].set_ylabel('Mean Squared Error')
axes[0].set_title('Training MSE always falls\nTest MSE has a U-shape sweet spot')
axes[0].legend()

# Right — log scale shows both curves clearly at all degrees
axes[1].semilogy(degrees, train_mse, 'o-', color='#1e5fa8', lw=2.5, label='Training MSE', markersize=7)
axes[1].semilogy(degrees, test_mse,  'o-', color='#e85d20', lw=2.5, label='Test MSE',     markersize=7)
axes[1].axvline(best_degree, color='#1a7a45', lw=2, ls='--',
                label=f'Sweet spot (degree {best_degree})')
axes[1].set_xticks(degrees)
axes[1].set_xlabel('Model Complexity (polynomial degree)')
axes[1].set_ylabel('MSE (log scale)')
axes[1].set_title('Log scale — see both curves clearly\nGap = overfitting')
axes[1].legend()

plt.suptitle('Bias-Variance Tradeoff: Training vs Test MSE', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n📌 Sweet spot: degree {best_degree} — lowest test MSE")
print(f"   Degree 1 : train={train_mse[0]:.3f}  test={test_mse[0]:.3f}  ← underfitting (high bias)")
print(f"   Degree {best_degree} : train={train_mse[best_degree-1]:.3f}  test={test_mse[best_degree-1]:.3f}  ← best generalisation ✓")
print(f"   Degree 8 : train={train_mse[-1]:.3f}  test={test_mse[-1]:.3f}  ← overfitting (high variance)")
print(f"\n   Key insight: Training MSE always falls — it can memorise anything.")
print(f"   Only test MSE reveals true generalisation ability.")


## 📊 Part 3 — Learning Curves — Diagnosing Bias vs Variance

In [ ]:
# ── Learning curves for underfit, good fit, overfit ─────────────────────────
# Define the true function and noise level (self-contained)
def f_true(x): return np.sin(2 * np.pi * x)
sigma = 0.3   # noise standard deviation — irreducible error floor = sigma² = 0.09

np.random.seed(1)
n_total = 200
X_full = np.random.uniform(0, 1, n_total)
Y_full = f_true(X_full) + np.random.normal(0, sigma, n_total)

train_sizes = range(10, 160, 10)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
model_specs = [(1, 'Degree 1 — Underfit (High Bias)'),
               (3, 'Degree 3 — Good Fit'),
               (8, 'Degree 8 — Overfit (High Variance)')]
model_colors = ['#e85d20', '#1a7a45', '#c0392b']

for ax, (deg, title), color in zip(axes, model_specs, model_colors):
    tr_errs, val_errs = [], []
    for n in train_sizes:
        X_tr, Y_tr = X_full[:n], Y_full[:n]
        X_val, Y_val = X_full[160:], Y_full[160:]
        pipe = Pipeline([('poly', PolynomialFeatures(deg)), ('lr', LinearRegression())])
        pipe.fit(X_tr.reshape(-1,1), Y_tr)
        tr_errs.append(mean_squared_error(Y_tr,   pipe.predict(X_tr.reshape(-1,1))))
        val_errs.append(mean_squared_error(Y_val, pipe.predict(X_val.reshape(-1,1))))

    ax.plot(list(train_sizes), tr_errs,  '-',  color=color,    lw=2, label='Training error')
    ax.plot(list(train_sizes), val_errs, '--', color='#1e5fa8', lw=2, label='Validation error')
    ax.axhline(sigma**2, color='#aaa', lw=1, ls=':', label=f'Irreducible floor (σ²={sigma**2})')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Training set size')
    if ax == axes[0]: ax.set_ylabel('MSE')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 0.35)

plt.suptitle('Learning Curves — Diagnosing Bias vs Variance', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("\n📌 How to read learning curves:")
print("   High bias  (left) : both curves plateau at HIGH error — model too simple")
print("   Good fit  (middle): both curves converge near the irreducible floor")
print("   High variance(right): big gap between train and validation — needs more data")
print(f"\n   Irreducible floor = σ² = {sigma**2:.2f} — the noise we can never remove")


## 💼 Exercise

Generate a dataset where the true function is `y = sin(2πx)` with noise σ=0.5 (n=50 train, n=200 test). Fit polynomial models of degree 1 through 10. Plot train vs test MSE. At what degree does overfitting begin? Confirm with the bias-variance decomposition.

In [ ]:
# ── Exercise workspace ──────────────────────────────────────────────────────
# Write your code here



In [ ]:
# @title 📝 Quiz — Bias-Variance Tradeoff
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** As model complexity increases, what happens to bias?
# @markdown **Q2:** As model complexity increases, what happens to variance?
# @markdown **Q3:** What does the irreducible error floor represent?
# @markdown **Q4:** A model with high training accuracy but poor test accuracy suffers from what?
# @markdown **Q5:** Name one practical way to reduce variance without changing the model family

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

In [ ]:
_NB_NAME_="Bias-Variance Tradeoff"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*